In this iteration, SMOTE was integrated into the imbalanced-learn pipeline to synthetically generate 8,000 critical tickets during training, paired with class_weight="balanced" on the Logistic Regression classifier. This produced a marginal improvement in precision and recall stability for critical predictions, but had no meaningful impact on the ROC/AUC curve. The underlying issue was that all critical tickets in the dataset are written in German, meaning SMOTE was only generating synthetic German critical samples — the model was effectively learning language as a proxy for priority rather than urgency of content, which is why no English tickets were ever predicted as critical.

In [1]:
from datasets import load_dataset
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

######## Run this if you want the untranslated dataset
# df = dataset["train"].to_pandas() 

######## Run this if you want the translated dataset
df = pd.read_csv("df_translated.csv")

/Users/kylebeedon/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Change very low to high
df.loc[df["priority"] == "very_low", "priority"] = "low"


# Initial cleaning

In [3]:
import re

def clean_text(text):
    text = text.lower()
    
    # remove common greeting prefixes at start of message
    text = re.sub(r"^(dear\s+)?(customer\s+(support|service)\s*(team)?|support\s*team)[\s,:\-]*", "", text)

    # remove generic greetings, urls, signnatures, and whitespace
    text = re.sub(r"^(hello|hi|greetings)[\s,:\-]*", "", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"(best regards|kind regards|sincerely).*", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# For english, change all df's to english

# Clean Subject and Body
df["body"] = df["body"].astype(str).apply(clean_text)
df["subject"] = df["subject"].astype(str).apply(clean_text)

# Concat subject and body into one "Text"
df["text"] = df["subject"] + " " + df["body"]

# Fill na with unknown
df["version"] = df["version"].fillna("unknown").astype(str)
# I doubt version would be a reasonable predictor but we shall see

# Combine all Tags into one
tag_cols = [f"tag_{i}" for i in range(1,9)]
df["all_tags"] = df[tag_cols].fillna("").agg(" ".join, axis=1)

# Train/Test split

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["priority"])
y = df["priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Preprocessor for Model

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

text_col = "text"
cat_cols = ["type", "version", "all_tags", "language"]

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=80000,
            ngram_range=(1, 3),
            stop_words="english",
            sublinear_tf=True,
            max_df=0.85,
            norm=None,
            min_df=5
        ), text_col),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# Pipeline using imblearn and SMOTE

In [7]:
from imblearn.pipeline import Pipeline 
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

# Create a sample of 8000 critical tickets
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("smote", SMOTE(
        sampling_strategy={"critical": 8000},
        random_state=42
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

In [8]:
pipeline.fit(X_train, y_train)


/Users/kylebeedon/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,steps,"[('preprocessing', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. T

# Evaluation 

In [9]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    critical       0.88      0.89      0.88       383
        high       0.79      0.79      0.79      4385
         low       0.80      0.77      0.78      2910
      medium       0.78      0.80      0.79      4675

    accuracy                           0.79     12353
   macro avg       0.81      0.81      0.81     12353
weighted avg       0.79      0.79      0.79     12353



# AUC/ROC score

In [10]:
from sklearn.metrics import roc_auc_score

auc_scores = roc_auc_score(
    y_test,
    pipeline.predict_proba(X_test),
    multi_class="ovr",
    average=None,
    labels=pipeline.classes_
)

print("Per-class AUC (One-vs-Rest):")
for cls, score in zip(pipeline.classes_, auc_scores):
    print(f"  {cls:>10}: {score:.3f}")

Per-class AUC (One-vs-Rest):
    critical: 0.998
        high: 0.917
         low: 0.932
      medium: 0.909


# Compute Probabilities

In [11]:
import numpy as np

y_probs = pipeline.predict_proba(X_test)
y_probs = np.round(y_probs, 4)

from sklearn.preprocessing import label_binarize

classes = pipeline.classes_
y_test_bin = label_binarize(y_test, classes=classes)

probs = pd.DataFrame(y_probs, columns=pipeline.classes_)
probs["max_prob"] = probs[pipeline.classes_].max(axis=1)
probs["true_label"] = y_test.values
probs["pred_label"] = y_pred

### TEST to see if model predicted any english tickets as critical

In [12]:
# Use logistic regression model to create eval df
eval_df = X_test.copy()

eval_df["y_true"] = y_test.values
eval_df["y_pred"] = y_pred
eval_df["correct"] = (eval_df["y_true"] == eval_df["y_pred"]).astype(int)

In [24]:
eval_df[(eval_df["y_pred"] == "critical") & (eval_df["language"].isin(["en"]))]

,subject,body,answer,type,queue,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text,all_tags,y_true,y_pred,correct


In [25]:
eval_df[(eval_df["y_pred"] == "critical") & (eval_df["language"].isin(["de"]))]

,subject,body,answer,type,queue,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text,all_tags,y_true,y_pred,correct
36413,sofortige unterstützung erforderlich: vorfall ...,"support-team,\n\nich schreibe ihnen, um eine k...",None,None,Pets & Animals/Pet Services,de,unknown,None,None,None,None,None,None,None,None,sofortige unterstützung erforderlich: vorfall ...,,critical,critical,1
34098,dringend: sicherheitsverletzung im personal ca...,"sehr geehrtes support-team,\n\nes wurde eine s...",None,None,Finance/Personal Finance,de,unknown,None,None,None,None,None,None,None,None,dringend: sicherheitsverletzung im personal ca...,,critical,critical,1
30850,dringend: geforce now serverausfall,"an das support-team,\n\nich melde ein kritisch...",None,None,Games,de,unknown,None,None,None,None,None,None,None,None,dringend: geforce now serverausfall an das sup...,,high,critical,0
31037,wichtig: pos-system ausgefallen - werbeaktion ...,"sehr geehrtes support-team,\n\nwir melden ein ...",None,None,Shopping/Retail Stores,de,unknown,None,None,None,None,None,None,None,None,wichtig: pos-system ausgefallen - werbeaktion ...,,critical,critical,1
31707,dringend: systemweiter ausfall mit auswirkunge...,"sehr geehrtes support-team,\n\nwir melden eine...",None,None,Jobs & Education/Recruitment,de,unknown,None,None,None,None,None,None,None,None,dringend: systemweiter ausfall mit auswirkunge...,,critical,critical,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29033,problem: snort-benachrichtigungen werden nicht...,"support-team,\n\nwir haben ein kritisches prob...",None,None,IT & Technology/Security Operations,de,unknown,None,None,None,None,None,None,None,None,problem: snort-benachrichtigungen werden nicht...,,high,critical,0
39286,benötigte unterstützung: dringender drm-fehler...,"sehr geehrtes support-team,\n\nwir haben ein k...",None,None,Books & Literature/Fiction,de,unknown,None,None,None,None,None,None,None,None,benötigte unterstützung: dringender drm-fehler...,,critical,critical,1
33593,reservierungen aufgrund von systemfehlfunktion...,"hallo, support-team. ich melde einen erheblich...",None,None,Travel & Transportation/Land Travel,de,unknown,None,None,None,None,None,None,None,None,reservierungen aufgrund von systemfehlfunktion...,,critical,critical,1
34388,dringend: fehlfunktion des spamfilters beeintr...,"sehr geehrte support-mitarbeiter,\n\nwir melde...",None,None,Online Communities/Forums,de,unknown,None,None,None,None,None,None,None,None,dringend: fehlfunktion des spamfilters beeintr...,,high,critical,0
